In [24]:
import sys
import pandas as pd
import joblib
import torch
from pathlib import Path
import numpy as np

code_dir = str(Path.cwd().parent / "src")

if code_dir not in sys.path:
    sys.path.append(code_dir)

from ledger_audit.evaluate import explain_xgboost, combine_reasons
from ledger_audit.rules import run_all_rules

In [25]:
# Go up one level from 'notebooks/' to the 'AuditLens/' root
root_dir = Path.cwd().parent

# Load test set features and scores
features_path = root_dir / "data/processed/features.csv"
labels_path = root_dir / "data/processed/labels.csv"
scores_path = root_dir / "data/processed/scores.csv"
ledger_path = root_dir / "data/raw/ledger_v3_with_anomalies.csv"
models_dir = root_dir / "data/processed/models"

# Read data
features = pd.read_csv(features_path, index_col="entry_id")
labels_full = pd.read_csv(labels_path, index_col="entry_id")
scores_df = pd.read_csv(scores_path, index_col="entry_id")
ledger = pd.read_csv(ledger_path)
ledger["posted_at"] = pd.to_datetime(ledger["posted_at"], utc=True)

# Filter to test set only
test_mask = scores_df["split"] == "test"
test_ids = scores_df[test_mask].index

X_test = features.loc[test_ids]
xgboost_scores = scores_df.loc[test_ids, "xgboost_score"]

print(f"Test set: {len(test_ids)} entries")
print(f"Features shape: {X_test.shape}")

Test set: 26395 entries
Features shape: (26395, 23)


In [26]:
# Load the trained XGBoost model
xgboost_model = joblib.load(models_dir / "xgboost.joblib")

print(f"Model loaded: {type(xgboost_model)}")
print(f"Model has {len(xgboost_model.get_booster().get_score())} features used")

Model loaded: <class 'xgboost.sklearn.XGBClassifier'>
Model has 21 features used


In [27]:
# Get SHAP explanations for top 20 entries by XGBoost score
top_20_indices = xgboost_scores.argsort(
)[-20:].values[::-1]  # Top 20, highest first
top_20_ids = test_ids[top_20_indices]
top_20_X = X_test.iloc[top_20_indices]

print(f"Getting SHAP explanations for top 20 entries...")
shap_reasons = explain_xgboost(xgboost_model, top_20_X, top_20_ids, top_n=3)

print(f"SHAP explanations generated for {len(shap_reasons)} entries")
print("\nExample SHAP reason:")
print(shap_reasons.iloc[0]["shap_reason"])

Getting SHAP explanations for top 20 entries...
SHAP explanations generated for 20 entries

Example SHAP reason:
is_weekend=1.0 (higher than typical, impact=14.980); user_hour_typicality=0.1 (lower than typical, impact=0.569); n_lines=2.0 (lower than typical, impact=0.517)


In [28]:
# Run all rules on the test ledger to get rule reasons
ledger_test = ledger[ledger["entry_id"].isin(test_ids)]

print(f"Running rules on test set ({len(ledger_test)} lines)...")
rule_findings = run_all_rules(ledger_test)

# Aggregate reasons by entry (combine multiple rules if they flagged the same entry)
rule_reasons = rule_findings.groupby("entry_id").agg({
    "reason": lambda x: " | ".join(x.unique()),
    "rule_name": lambda x: ", ".join(x.unique())
}).reset_index()
rule_reasons.rename(columns={"rule_name": "rules_fired"}, inplace=True)

print(f"Rule findings: {len(rule_reasons)} entries flagged by rules")
print("\nExample rule reason:")
if len(rule_reasons) > 0:
    print(rule_reasons.iloc[0]["reason"])

Running rules on test set (59449 lines)...
Rule findings: 281 entries flagged by rules

Example rule reason:
The net expense amount is just below the £10,000 approval limit.


In [29]:
# Filter rule reasons to only top 20 entries
rule_reasons_top20 = rule_reasons[rule_reasons["entry_id"].isin(
    top_20_ids)].copy()

# Rename for combine_reasons function
rule_reasons_top20_renamed = rule_reasons_top20[["entry_id", "reason"]].copy()

# Combine rule and SHAP reasons
combined = combine_reasons(rule_reasons_top20_renamed, shap_reasons)

# For entries with no rules, still keep the SHAP reason
all_top20_combined = pd.DataFrame({"entry_id": top_20_ids})
all_top20_combined = all_top20_combined.merge(
    combined, on="entry_id", how="left")
all_top20_combined["why_flagged"] = all_top20_combined["why_flagged"].fillna(
    "Model flagged based on feature patterns")

print(f"Combined explanations for {len(all_top20_combined)} entries")

Combined explanations for 20 entries


In [30]:
# Get scores and other details for top 20
results_df = pd.DataFrame({
    "entry_id": top_20_ids,
    "xgboost_score": xgboost_scores.loc[top_20_ids].values,
})

# Add if entry is actually anomalous
results_df["is_anomaly"] = results_df["entry_id"].map(
    labels_full["is_anomaly"].to_dict()
)

# Add anomaly type
results_df["anomaly_type"] = results_df["entry_id"].map(
    labels_full["anomaly_type"].to_dict()
)

# Add why_flagged explanation
results_df = results_df.merge(all_top20_combined[["entry_id", "why_flagged"]],
                              on="entry_id", how="left")

# Reorder columns for readability
results_df = results_df[[
    "entry_id",
    "xgboost_score",
    "is_anomaly",
    "anomaly_type",
    "why_flagged"
]]

results_df = results_df.sort_values(
    "xgboost_score", ascending=False).reset_index(drop=True)

print("Top 20 entries by XGBoost score:")
print(results_df)

Top 20 entries by XGBoost score:
    entry_id  xgboost_score  is_anomaly       anomaly_type  \
0      64839       0.999998        True         odd_timing   
1     131386       0.999997        True         odd_timing   
2      59402       0.999995        True         odd_timing   
3     114388       0.999995        True         odd_timing   
4     180146       0.999995        True  duplicate_payment   
5      50797       0.999990        True         odd_timing   
6     180169       0.999990        True  duplicate_payment   
7      81646       0.999989        True         odd_timing   
8      76258       0.999989        True         odd_timing   
9     180037       0.999982        True  duplicate_payment   
10     81096       0.999981        True         odd_timing   
11    180226       0.999980        True  duplicate_payment   
12     83476       0.999978        True         odd_timing   
13    127564       0.999972        True         odd_timing   
14    153155       0.999957        Tr

In [31]:
# Display with better formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

print("\n" + "="*150)
print("TOP 20 FLAGGED ENTRIES WITH EXPLANATIONS")
print("="*150 + "\n")

for idx, row in results_df.iterrows():
    print(f"{idx+1}. Entry ID: {row['entry_id']}")
    print(f"   XGBoost Score: {row['xgboost_score']:.4f}")
    print(
        f"   Actual Anomaly: {'YES' if row['is_anomaly'] == 1 else 'NO'} ({row['anomaly_type']})")
    print(f"   Why Flagged: {row['why_flagged']}")
    print()


TOP 20 FLAGGED ENTRIES WITH EXPLANATIONS

1. Entry ID: 64839
   XGBoost Score: 1.0000
   Actual Anomaly: YES (odd_timing)
   Why Flagged: Rule: The entry was posted on a weekend or outside allowed hours. Model: is_weekend=1.0 (higher than typical, impact=14.980); user_hour_typicality=0.1 (lower than typical, impact=0.569); n_lines=2.0 (lower than typical, impact=0.517)

2. Entry ID: 131386
   XGBoost Score: 1.0000
   Actual Anomaly: YES (odd_timing)
   Why Flagged: Rule: The entry was posted on a weekend or outside allowed hours. Model: is_weekend=1.0 (higher than typical, impact=15.078); total_amount_pence=52204.0 (lower than typical, impact=1.084); user_hour_typicality=0.1 (lower than typical, impact=0.742)

3. Entry ID: 59402
   XGBoost Score: 1.0000
   Actual Anomaly: YES (odd_timing)
   Why Flagged: Rule: The entry was posted on a weekend or outside allowed hours. Model: is_weekend=1.0 (higher than typical, impact=15.234); total_amount_pence=92033.0 (lower than typical, impact=0.

In [32]:
# How many of the top 20 are actual anomalies?
correct = results_df["is_anomaly"].sum()
accuracy = correct / len(results_df) * 100

print(f"\nAccuracy of top 20:")
print(f"  {correct} out of {len(results_df)} are actual anomalies ({accuracy:.1f}%)")
print(f"\nAnomalies found by type:")
print(results_df[results_df["is_anomaly"] == 1]["anomaly_type"].value_counts())


Accuracy of top 20:
  20 out of 20 are actual anomalies (100.0%)

Anomalies found by type:
anomaly_type
odd_timing           14
duplicate_payment     6
Name: count, dtype: int64


In [33]:
# Save the results for future reference
output_path = root_dir / "data/processed/top_20_explained.csv"
results_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to /Users/greatdaveo/CODE_BASE/ML-and-AI/audit-detection/AuditLens/data/processed/top_20_explained.csv
